## Recommendation System

#### Data Description:

Unique ID of each anime.
    
Anime title.
    
Anime broadcast type, such as TV, OVA, etc.
    
anime genre.
    
The number of episodes of each anime.
    
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.
    
#### Objective:
    
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
    
#### Dataset:
    
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

#### Tasks:


##### Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
    
Handle missing values, if any.
    
Explore the dataset to understand its structure and attributes.


In [1]:
import pandas as pd

# 1. Load the dataset
file_path = r"C:\Users\Rakshitha\Downloads\anime.csv"
df = pd.read_csv(file_path)

# 2. Display basic information about the dataset
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

# 3. Check for missing values
print("\nMissing Values in Each Column:")
print(df.isnull().sum())

# 4. Handling missing values
# Option 1: Drop rows with missing values
df_cleaned = df.dropna()

# Option 2: Fill missing values (example)
# df['column_name'].fillna(df['column_name'].mean(), inplace=True)

print("\nAfter Handling Missing Values:")
print(df_cleaned.isnull().sum())

# 5. Explore dataset structure
print("\nColumn Names:")
print(df.columns)

print("\nStatistical Summary:")
print(df.describe(include='all'))

# 6. Check data types
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (12294, 7)

First 5 rows:
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (

##### Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
    
Convert categorical features into numerical representations if necessary.
    
Normalize numerical features if required.


In [2]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

# Load dataset
file_path = r"C:\Users\Rakshitha\Downloads\anime.csv"
df = pd.read_csv(file_path)


# 1. Select relevant features

# Common useful columns: 'name', 'genre', 'rating', 'members'
df = df[['name', 'genre', 'rating', 'members']]


# 2. Handle missing values

df = df.dropna(subset=['genre', 'rating'])

# Fill missing numeric values if any
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['members'] = df['members'].fillna(df['members'].median())


# 3. Convert categorical features (Genre → numerical)

# Genres are usually separated by commas
df['genre'] = df['genre'].apply(lambda x: x.split(', '))

mlb = MultiLabelBinarizer()
genre_encoded = pd.DataFrame(mlb.fit_transform(df['genre']),
                             columns=mlb.classes_,
                             index=df.index)


# 4. Normalize numerical features

scaler = MinMaxScaler()
df[['rating', 'members']] = scaler.fit_transform(df[['rating', 'members']])


# 5. Combine all features

features = pd.concat([df[['rating', 'members']], genre_encoded], axis=1)

print("Feature matrix shape:", features.shape)
print("\nSample features:")
print(features.head())

Feature matrix shape: (12017, 45)

Sample features:
     rating   members  Action  Adventure  Cars  Comedy  Dementia  Demons  \
0  0.924370  0.197867       0          0     0       0         0       0   
1  0.911164  0.782769       1          1     0       0         0       0   
2  0.909964  0.112683       1          0     0       1         0       0   
3  0.900360  0.664323       0          0     0       0         0       0   
4  0.899160  0.149180       1          0     0       1         0       0   

   Drama  Ecchi  ...  Shounen Ai  Slice of Life  Space  Sports  Super Power  \
0      1      0  ...           0              0      0       0            0   
1      1      0  ...           0              0      0       0            0   
2      0      0  ...           0              0      0       0            0   
3      0      0  ...           0              0      0       0            0   
4      0      0  ...           0              0      0       0            0   

   Supernatural 

##### Recommendation System:

Design a function to recommend anime based on cosine similarity.
    
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
    
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
    
Analyze the performance of the recommendation system and identify areas of improvement.


In [4]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity


# 1. Load dataset

file_path = r"C:\Users\Rakshitha\Downloads\anime.csv"
df = pd.read_csv(file_path)


# 2. Select relevant columns

df = df[['name', 'genre', 'rating', 'members']]


# 3. Handle missing values

df = df.dropna(subset=['genre', 'rating'])

df['rating'] = df['rating'].fillna(df['rating'].mean())
df['members'] = df['members'].fillna(df['members'].median())


# 4. Convert genres into numerical format

df['genre'] = df['genre'].apply(lambda x: x.split(', '))

mlb = MultiLabelBinarizer()
genre_encoded = pd.DataFrame(mlb.fit_transform(df['genre']),
                             columns=mlb.classes_,
                             index=df.index)


# 5. Normalize numerical features

scaler = MinMaxScaler()
df[['rating', 'members']] = scaler.fit_transform(df[['rating', 'members']])


# 6. Combine all features

features = pd.concat([df[['rating', 'members']], genre_encoded], axis=1)


# 7. Compute cosine similarity

similarity_matrix = cosine_similarity(features)


# 8. Recommendation function

def recommend_anime(anime_name, threshold=0.5, top_n=10):
    if anime_name not in df['name'].values:
        return "Anime not found in dataset."
    
    # Get index of the anime
    idx = df[df['name'] == anime_name].index[0]
    
    # Get similarity scores
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Sort by similarity score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Filter based on threshold and exclude itself
    recommendations = []
    for i, score in sim_scores[1:]:
        if score >= threshold:
            recommendations.append((df.iloc[i]['name'], score))
    
    # Limit top N results
    return recommendations[:top_n]


# 9. Example usage

anime_name = "Naruto"  # Change to any anime name in dataset

recommendations = recommend_anime(anime_name, threshold=0.5, top_n=10)

print(f"\nRecommendations for '{anime_name}':\n")
for name, score in recommendations:
    print(f"{name} - Similarity Score: {score:.4f}")


Recommendations for 'Naruto':

Naruto: Shippuuden - Similarity Score: 0.9982
Naruto: Shippuuden Movie 4 - The Lost Tower - Similarity Score: 0.9705
Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono - Similarity Score: 0.9704
Boruto: Naruto the Movie - Similarity Score: 0.9694
Naruto x UT - Similarity Score: 0.9640
Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!! - Similarity Score: 0.9637
Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi - Similarity Score: 0.9633
Naruto Shippuuden: Sunny Side Battle - Similarity Score: 0.9625
Katekyo Hitman Reborn! - Similarity Score: 0.8963
Kyutai Panic Adventure! - Similarity Score: 0.8615


#### Interview Questions:


##### 1. Can you explain the difference between user-based and item-based collaborative filtering?

##### User-Based Collaborative Filtering

-> Recommends items based on similar users.
    
-> Idea: “Users who are similar to you liked these items.”

##### How it works:

Find users with similar preferences (based on ratings/history).
    
Look at what those similar users liked.
    
Recommend those items to the target user.
    
##### Example:
    
If User A and User B both liked similar anime,

And User B watched a new anime,

That anime is recommended to User A.
    
##### Pros:

Personalized recommendations

Captures user taste patterns

##### Cons:

Struggles with scalability (many users)

Cold start problem for new users

##### Item-Based Collaborative Filtering

Recommends items based on similar items.
    
Idea: “Users who liked this item also liked these similar items.”

##### How it works:
    
Compute similarity between items (based on user ratings).
    
For a given item, find similar items.
    
Recommend those similar items.
    
##### Example:

If users who watched Naruto also watched Bleach,

Then Bleach is recommended when someone watches Naruto.
    
##### Pros:

More scalable than user-based

Stable (item relationships don’t change frequently)

Works well in large systems

##### Cons:

Less personalized compared to user-based

Needs enough interaction data for items

##### 2. What is collaborative filtering, and how does it work?

##### ->

Collaborative filtering is a recommendation technique used in systems (like Netflix, Amazon, or anime platforms) that suggests items to a user based on the preferences and behavior of other users.

##### ->

Collaborative filtering works by analyzing a user–item interaction matrix, which typically contains:

Users
    
Items (movies, anime, products, etc.)
    
Ratings or interactions (likes, views, clicks)